# Phase 03 - Classifier core

Datasets and transforms, model builders (ResNet50, ConvNeXt-Tiny, Swin-Tiny), the
train/evaluate loop and run folders. See `docs/instructions/03-classifier-core.md`.

Cells whose first line is `#| export` are copied, in order, into the
generated `group3-starvenus.py` by `tools/export_py.py`. Keep outputs cleared before committing.

The experiment registry (`EXPERIMENTS`) and the choices file live in `90_cli.ipynb`; this notebook is
the engine that the registry drives.

## Experiment configuration (T3.1)

One `ExperimentConfig` describes one row of the experiment matrix (README section 5). The engine below
is generic: it never looks at `exp_id`, only at these fields.

Three fields can be `None` in a registry entry and are then filled in **at run time** from
`_work/results/choices.json` (written by the gates of phase 04): `input_mode` ("best of E0"),
`backbone` ("best of E2") and `lr` (per backbone, from the split-0 sweep). A resolved config has none
of them left empty.

`select_domains` always equals `train_domains`: the checkpoint is chosen on the validation images of
the domains the model was trained on. A white-only model (W->F rows) is therefore chosen on **white**
validation images; choosing on field would leak field information into a "white-only" model.

In [ ]:
#| export
import gc
import math
import platform
import random
import re
import time
from urllib.parse import urlparse

import torch
import torch.nn.functional as F
import torchvision
from sklearn.metrics import confusion_matrix, f1_score, precision_recall_fscore_support

NUM_CLASSES = len(CLASS_NAMES)
BACKBONES = ("resnet50", "convnext_tiny", "swin_t")
INPUT_MODES = ("squash", "pad", "rect")
# (height, width) of the network input at an equal pixel budget (~100k px): E0 compares geometry, not resolution.
INPUT_SIZES = {"squash": (320, 320), "pad": (320, 320), "rect": (448, 224)}
SUBSETS = ("white", "field", "mixed")
KINDS = ("classifier", "segmenter", "analysis")


class ExperimentError(Exception):
    """Base class of errors that the CLI reports as one clean line (no traceback)."""


class UnknownExperimentError(ExperimentError):
    pass


class ChoicesError(ExperimentError):
    """A value that a gate should have written to choices.json is missing."""


class NotReadyError(ExperimentError):
    """The experiment needs a later phase that is not implemented yet."""


@dataclass(frozen=True)
class ExperimentConfig:
    exp_id: str
    backbone: str | None  # None: the winning backbone of gate G1 (choices.json "backbone")
    train_domains: tuple[str, ...]
    select_domains: tuple[str, ...]  # must equal train_domains
    test_subsets: tuple[str, ...]  # which test subsets the report uses (all three are always evaluated)
    input_mode: str | None  # None: the winning input mode of gate G0 (choices.json "input_mode")
    use_composites: bool = False  # Method A: extra training samples from CompositeSource (phase 05)
    seg_crop: bool = False  # Method B: input_hook crops to the segmenter box (phase 06)
    kind: str = "classifier"  # "segmenter" / "analysis" rows exist in the matrix but are not run here
    alias_of_winner: str = ""  # "E0": this row is the winning E0 row (same run folder, no retraining)
    # Recipe (section 6.1 of the report). Defaults are the T3.4 table.
    stage1_epochs: int = 2  # linear probe: backbone frozen, head only
    stage1_lr: float = 1e-3
    epochs: int = 30  # stage 2: fine-tune everything
    lr: float | None = None  # None: choices.json lr[backbone] (split-0 sweep over SWEEP_LRS)
    weight_decay: float = 0.05
    warmup_epochs: int = 1
    label_smoothing: float = 0.1
    batch_size: int = 32
    grad_accum: int = 1
    clip_norm: float = 1.0
    patience: int = 10  # stage-2 epochs without improvement of the selection metric
    dropout: float = 0.2
    pretrained: bool = True
    size_div: int = 1  # dev/smoke only: divide the input size (448x224 -> 112x56 with 4)
    note: str = ""

    def __post_init__(self):
        if self.kind not in KINDS:
            raise ValueError(f"{self.exp_id}: kind must be one of {KINDS}, got {self.kind!r}")
        if self.backbone is not None and self.backbone not in BACKBONES and self.kind == "classifier":
            raise ValueError(f"{self.exp_id}: unknown backbone {self.backbone!r}")
        if self.input_mode is not None and self.input_mode not in INPUT_MODES:
            raise ValueError(f"{self.exp_id}: unknown input mode {self.input_mode!r}")
        for name in ("train_domains", "select_domains"):
            domains = getattr(self, name)
            if not domains or not set(domains) <= set(DOMAINS):
                raise ValueError(f"{self.exp_id}: {name} must be a non-empty subset of {DOMAINS}, got {domains!r}")
        if set(self.select_domains) != set(self.train_domains):
            raise ValueError(
                f"{self.exp_id}: select_domains must equal train_domains (checkpoints are chosen on the domains "
                f"the model was trained on; anything else leaks information), got {self.select_domains} vs {self.train_domains}"
            )
        if not set(self.test_subsets) <= set(SUBSETS):
            raise ValueError(f"{self.exp_id}: test_subsets must be a subset of {SUBSETS}")

    def is_resolved(self) -> bool:
        return self.backbone is not None and self.input_mode is not None and self.lr is not None

## Dataset and transforms (T3.2)

No PIL: images come from the 1280 px cache (`read_cached_rgb`) as `uint8` arrays and are processed with
OpenCV and numpy. A sample is returned as a **`uint8` CHW tensor**; normalisation with the ImageNet mean/std
happens on the compute device (`normalize_batch`), which moves four times fewer bytes between worker
processes and the training loop. Dataset statistics are never used, because they would include test images.

**Input modes** (equal pixel budget, ~100k px, so E0 compares geometry and not resolution):
`squash` resizes to 320x320 (aspect distorted ~2x), `pad` letterboxes to 320x320 with the ImageNet mean
colour (aspect kept), `rect` resizes to 448x224 (H x W), which is close to the images' 0.47 aspect.

**Training augmentation**, in this order: random crop of 60-100% of the area keeping the image's own aspect,
horizontal and vertical flip (p = 0.5 each), rotation +-15 degrees with reflected borders, resize into the
input mode, then colour jitter (brightness 0.2, contrast 0.2, saturation 0.15, hue 0.02 only: colour is
diagnostic, e.g. Tungro is yellow-orange). Jitter is applied after the resize purely for cost; it is pointwise,
so it commutes with resizing, and in `pad` mode it must not touch the letterbox colour anyway.

**Determinism with workers.** Every random draw of a sample comes from `np.random.default_rng([seed, epoch,
index])`, built inside `__getitem__`. Nothing depends on which worker process handles the sample or on
the order of loading, so a run is reproducible for any `num_workers`. The `EpochSampler` hands each worker
`(index, epoch)` pairs, which also lets us keep the workers alive between epochs (`persistent_workers`),
important under `spawn` (macOS, Windows) where starting a worker re-imports torch.

**Hooks.** `extra_sources` is a sequence of objects with `__len__` and `load(index, epoch) -> (rgb, class_idx,
name)`; Method A's `CompositeSource` (phase 05) plugs in there and passes through the same augmentation.
`input_hook(image, record)` runs on real images before the transform, at train and eval time alike; Method B's
crop (phase 06) plugs in there. `record` is `None` for ad-hoc images (the `predict` command).

In [ ]:
#| export
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
PAD_COLOR = tuple(round(m * 255) for m in IMAGENET_MEAN)  # letterbox colour: the ImageNet mean, (124, 116, 104)

AUG_CROP_AREA = (0.6, 1.0)  # fraction of the image area kept by the random crop
AUG_ROTATION_DEG = 15.0
AUG_BRIGHTNESS = 0.2
AUG_CONTRAST = 0.2
AUG_SATURATION = 0.15
AUG_HUE = 0.02  # fraction of the hue circle: +-7.2 degrees at most


def input_size(mode: str, size_div: int = 1) -> tuple[int, int]:
    """(height, width) of the network input for an input mode. ``size_div`` > 1 is for smoke tests only."""
    if mode not in INPUT_SIZES:
        raise ValueError(f"unknown input mode {mode!r}, expected one of {INPUT_MODES}")
    height, width = INPUT_SIZES[mode]
    return max(8, height // size_div), max(8, width // size_div)


def resize_exact(rgb: np.ndarray, height: int, width: int) -> np.ndarray:
    shrinking = height * width < rgb.shape[0] * rgb.shape[1]
    return cv2.resize(rgb, (width, height), interpolation=cv2.INTER_AREA if shrinking else cv2.INTER_LINEAR)


def resize_content(rgb: np.ndarray, mode: str, size: tuple[int, int]) -> np.ndarray:
    """The image content at network resolution. For `pad` this is the aspect-preserving fit inside the
    box (smaller than the box in one dimension); `letterbox` then places it on the canvas."""
    height, width = size
    if mode != "pad":
        return resize_exact(rgb, height, width)
    scale = min(height / rgb.shape[0], width / rgb.shape[1])
    return resize_exact(rgb, max(1, round(rgb.shape[0] * scale)), max(1, round(rgb.shape[1] * scale)))


def letterbox(content: np.ndarray, size: tuple[int, int]) -> np.ndarray:
    height, width = size
    if content.shape[:2] == (height, width):
        return content
    canvas = np.empty((height, width, 3), dtype=np.uint8)
    canvas[:] = PAD_COLOR
    top, left = (height - content.shape[0]) // 2, (width - content.shape[1]) // 2
    canvas[top : top + content.shape[0], left : left + content.shape[1]] = content
    return canvas


def random_crop(rgb: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    """Crop covering 60-100% of the area at the image's own aspect (a zoom / framing change)."""
    height, width = rgb.shape[:2]
    side = math.sqrt(rng.uniform(*AUG_CROP_AREA))
    crop_h, crop_w = max(1, round(height * side)), max(1, round(width * side))
    top = int(rng.integers(0, height - crop_h + 1))
    left = int(rng.integers(0, width - crop_w + 1))
    return rgb[top : top + crop_h, left : left + crop_w]


def random_flips(rgb: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    flip_h, flip_v = rng.random() < 0.5, rng.random() < 0.5
    if flip_h:
        rgb = rgb[:, ::-1]
    if flip_v:
        rgb = rgb[::-1]
    return np.ascontiguousarray(rgb)


def random_rotation(rgb: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    angle = rng.uniform(-AUG_ROTATION_DEG, AUG_ROTATION_DEG)
    height, width = rgb.shape[:2]
    matrix = cv2.getRotationMatrix2D((width / 2, height / 2), angle, 1.0)
    return cv2.warpAffine(rgb, matrix, (width, height), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT_101)


def _gray(img: np.ndarray) -> np.ndarray:
    return img[..., 0] * 0.299 + img[..., 1] * 0.587 + img[..., 2] * 0.114


def color_jitter(rgb: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    """Brightness, contrast, saturation (multiplicative factors) and a small hue shift, in random order."""
    brightness = rng.uniform(1 - AUG_BRIGHTNESS, 1 + AUG_BRIGHTNESS)
    contrast = rng.uniform(1 - AUG_CONTRAST, 1 + AUG_CONTRAST)
    saturation = rng.uniform(1 - AUG_SATURATION, 1 + AUG_SATURATION)
    hue = rng.uniform(-AUG_HUE, AUG_HUE)
    order = rng.permutation(4)  # all draws happen up front so the stream does not depend on the order
    img = rgb.astype(np.float32) / 255.0
    for op in order:
        if op == 0:
            img = np.clip(img * brightness, 0.0, 1.0)
        elif op == 1:
            mean = float(_gray(img).mean())
            img = np.clip((img - mean) * contrast + mean, 0.0, 1.0)
        elif op == 2:
            gray = _gray(img)[..., None]
            img = np.clip((img - gray) * saturation + gray, 0.0, 1.0)
        else:
            hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)  # float32: H in [0, 360), S and V in [0, 1]
            hsv[..., 0] = (hsv[..., 0] + hue * 360.0) % 360.0
            img = np.clip(cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB), 0.0, 1.0)
    return np.clip(img * 255.0 + 0.5, 0, 255).astype(np.uint8)


@dataclass(frozen=True)
class ImageTransform:
    """uint8 RGB image -> uint8 RGB image of the network input size. ``train`` adds the augmentation (and
    needs an ``rng``); the evaluation transform is the input-mode resize only and is deterministic."""

    mode: str
    train: bool = False
    size_div: int = 1

    @property
    def size(self) -> tuple[int, int]:
        return input_size(self.mode, self.size_div)

    def __call__(self, rgb: np.ndarray, rng: np.random.Generator | None = None) -> np.ndarray:
        if self.train:
            if rng is None:
                raise ValueError("the training transform needs an rng")
            rgb = random_rotation(random_flips(random_crop(rgb, rng), rng), rng)
        content = resize_content(np.ascontiguousarray(rgb), self.mode, self.size)
        if self.train:
            content = color_jitter(content, rng)
        return letterbox(content, self.size)


def normalize_batch(images: torch.Tensor, device) -> torch.Tensor:
    """uint8 [N, 3, H, W] -> float32 on ``device``, scaled to [0, 1] and normalised with the ImageNet mean/std."""
    x = images.to(device, non_blocking=True).float().div_(255.0)
    mean = torch.tensor(IMAGENET_MEAN, device=x.device).view(1, 3, 1, 1)
    std = torch.tensor(IMAGENET_STD, device=x.device).view(1, 3, 1, 1)
    return (x - mean) / std


def select_records(records: list[Record], filenames, domains=None) -> list[Record]:
    """Records of a split part (``filenames``, e.g. split['train']) restricted to ``domains``, in split order."""
    by_name = {r.filename: r for r in records}
    missing = [name for name in filenames if name not in by_name]
    if missing:
        raise DatasetError(f"{len(missing)} split entries are not in the index, e.g. {missing[:3]}; re-run `prepare`")
    chosen = [by_name[name] for name in filenames]
    return [r for r in chosen if domains is None or r.domain in domains]


def subsample_records(records: list[Record], n: int) -> list[Record]:
    """At most ``n`` records, round-robin over the class x domain strata (deterministic). For smoke runs."""
    queues: dict = {}
    for record in records:
        queues.setdefault(record.stratum, []).append(record)
    order = sorted(queues, key=stratum_order)
    chosen: list[Record] = []
    while len(chosen) < min(n, len(records)):
        for stratum in order:
            if queues[stratum] and len(chosen) < n:
                chosen.append(queues[stratum].pop(0))
    return chosen


def _sample_seed(seed: int) -> int:
    return int(seed) & 0x7FFFFFFF


class LeafDataset:
    """Map-style dataset over cached images.

    ``dataset[i]`` and ``dataset[(i, epoch)]`` both return ``(uint8 CHW tensor, class index, i)``. The
    augmentation of a sample is a pure function of ``(seed, epoch, i)``. Indices ``0..len(records)-1`` are
    the real records; the extra sources follow.
    """

    def __init__(self, data_root, records, transform, *, seed=0, extra_sources=(), input_hook=None, epoch=0):
        self.data_root = Path(data_root)
        self.records = list(records)
        self.transform = transform
        self.seed = _sample_seed(seed)
        self.extra_sources = tuple(extra_sources)
        self.input_hook = input_hook
        self.epoch = epoch
        self._extra_offsets = np.cumsum([0, *[len(s) for s in self.extra_sources]])

    def __len__(self) -> int:
        return len(self.records) + int(self._extra_offsets[-1])

    def sample_rng(self, index: int, epoch: int) -> np.random.Generator:
        return np.random.default_rng([self.seed, int(epoch), int(index)])

    def load_raw(self, index: int, epoch: int = 0) -> tuple[np.ndarray, int, str]:
        """(uint8 RGB before the transform, class index, name) after the hook for real images."""
        if index < len(self.records):
            record = self.records[index]
            rgb = read_cached_rgb(self.data_root, record.filename)
            if self.input_hook is not None:
                rgb = self.input_hook(rgb, record)
            return rgb, record.cls_idx, record.filename
        extra = index - len(self.records)
        which = int(np.searchsorted(self._extra_offsets, extra, side="right") - 1)
        return self.extra_sources[which].load(extra - int(self._extra_offsets[which]), epoch)

    def __getitem__(self, key):
        index, epoch = key if isinstance(key, tuple) else (key, self.epoch)
        if not 0 <= index < len(self):
            raise IndexError(index)
        rgb, label, _ = self.load_raw(index, epoch)
        rng = self.sample_rng(index, epoch) if self.transform.train else None
        image = self.transform(rgb, rng)
        return torch.from_numpy(np.ascontiguousarray(image.transpose(2, 0, 1))), int(label), index


class EpochSampler(torch.utils.data.Sampler):
    """Shuffled ``(index, epoch)`` pairs; the order is a function of (seed, epoch) only."""

    def __init__(self, n: int, seed: int):
        self.n, self.seed, self.epoch = n, _sample_seed(seed), 0

    def set_epoch(self, epoch: int) -> None:
        self.epoch = epoch

    def __len__(self) -> int:
        return self.n

    def __iter__(self):
        order = np.random.default_rng([self.seed, self.epoch]).permutation(self.n)
        return iter([(int(i), self.epoch) for i in order])


def _worker_init(worker_id: int) -> None:
    cv2.setNumThreads(0)  # OpenCV's own thread pool would fight with the DataLoader workers
    torch.set_num_threads(1)


def default_num_workers() -> int:
    """Data-loading processes, at most 4. Decoding and augmenting one image costs ~30 ms of CPU, so on a 2-vCPU
    Colab machine the workers, not the GPU, set the speed; the main process mostly waits for the GPU there, so
    every core gets a worker. From 3 cores up, one core is left to the main process."""
    try:
        cores = len(os.sched_getaffinity(0))
    except AttributeError:
        cores = os.cpu_count() or 1
    return max(1, min(4, cores if cores <= 2 else cores - 1))


def usable_workers(requested: int) -> int:
    """Workers that can really start: under `spawn` they re-import this file, which only works when it runs
    as the main script (the same rule as the cache's process pool)."""
    return requested if requested > 0 and pool_is_safe() else 0


def make_loader(dataset, batch_size, *, sampler=None, num_workers=0, drop_last=False, pin_memory=False, mp_context=None):
    num_workers = usable_workers(num_workers) if mp_context is None else num_workers
    kwargs = {}
    if num_workers > 0:
        kwargs.update(
            persistent_workers=True, worker_init_fn=_worker_init, multiprocessing_context=mp_context, prefetch_factor=4
        )
    return torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        sampler=sampler,
        shuffle=False,
        num_workers=num_workers,
        drop_last=drop_last,
        pin_memory=pin_memory,
        **kwargs,
    )


@dataclass
class EvalSet:
    """A fully materialised evaluation set: the deterministic eval transform applied once, kept as uint8."""

    images: torch.Tensor  # [N, 3, H, W] uint8
    labels: np.ndarray  # [N] int64
    names: list[str]
    domains: list[str]

    def __len__(self) -> int:
        return len(self.names)


def materialize(dataset: LeafDataset, *, num_workers: int = 0, batch_size: int = 32) -> EvalSet:
    """Run the (deterministic) eval transform over a dataset once. Per-epoch validation then costs only the
    forward pass instead of decoding JPEGs every epoch."""
    if len(dataset) != len(dataset.records):
        raise ValueError("materialize() is for real records only (no extra_sources)")
    loader = make_loader(dataset, batch_size, num_workers=num_workers)
    batches = [images for images, _, _ in loader]
    images = torch.cat(batches) if batches else torch.empty(0, 3, *dataset.transform.size, dtype=torch.uint8)
    return EvalSet(
        images,
        np.array([r.cls_idx for r in dataset.records], dtype=np.int64),
        [r.filename for r in dataset.records],
        [r.domain for r in dataset.records],
    )

## Models (T3.3)

Three torchvision backbones with ImageNet-pretrained weights: ResNet50 (`IMAGENET1K_V2`), ConvNeXt-Tiny
and Swin-Tiny (`IMAGENET1K_V1`). Only the final layer is replaced, by `Dropout(0.2) + Linear(in, 5)`.
Swin-T and ConvNeXt-T have ~28M parameters each, so "CNN vs transformer" is a controlled comparison.

* `head_parameters` / `backbone_parameters` split the parameters for the linear-probe stage (head only).
* `gradcam_target_layer` names the layer whose activations Grad-CAM (phase 07) uses: ResNet `layer4`,
  ConvNeXt `features[-1]`, Swin `features[-1]`. **Swin returns NHWC** (batch, height, width, channels)
  from that layer, while Grad-CAM wants NCHW, so `cam_to_nchw` permutes it.
* Swin-T accepts 448x224 and 320x320 although its windows are 7x7 tokens: torchvision pads each stage to a
  multiple of the window size internally.

In [ ]:
#| export
from torch import nn


@dataclass(frozen=True)
class BackboneSpec:
    builder: str  # function in torchvision.models
    weights: str  # "<Weights enum>.<member>", resolved with torchvision.models.get_weight


BACKBONE_SPECS = {
    "resnet50": BackboneSpec("resnet50", "ResNet50_Weights.IMAGENET1K_V2"),
    "convnext_tiny": BackboneSpec("convnext_tiny", "ConvNeXt_Tiny_Weights.IMAGENET1K_V1"),
    "swin_t": BackboneSpec("swin_t", "Swin_T_Weights.IMAGENET1K_V1"),
}
# Weights that `download-weights` fetches: (name, weights enum member, what it is for).
SEGMENTER_WEIGHTS = "LRASPP_MobileNet_V3_Large_Weights.COCO_WITH_VOC_LABELS_V1"
WEIGHT_TABLE = (
    ("resnet50", BACKBONE_SPECS["resnet50"].weights, "ResNet50 classifier (E0, E1)"),
    ("convnext_tiny", BACKBONE_SPECS["convnext_tiny"].weights, "ConvNeXt-Tiny classifier (E2)"),
    ("swin_t", BACKBONE_SPECS["swin_t"].weights, "Swin-Tiny classifier (E2)"),
    ("lraspp_mobilenet_v3_large", SEGMENTER_WEIGHTS, "leaf segmenter (phase 06)"),
)


def backbone_weights(backbone: str):
    return torchvision.models.get_weight(BACKBONE_SPECS[backbone].weights)


def backbone_of(model: nn.Module) -> str:
    models = torchvision.models
    for name, cls in (("resnet50", models.ResNet), ("convnext_tiny", models.ConvNeXt), ("swin_t", models.SwinTransformer)):
        if isinstance(model, cls):
            return name
    raise ValueError(f"not one of our backbones: {type(model).__name__}")


def build_model(backbone: str, num_classes: int = NUM_CLASSES, *, pretrained: bool = True, dropout: float = 0.2, data_root=None):
    """torchvision backbone with the final layer replaced by Dropout + Linear. ``pretrained=False`` (tests,
    ``predict``) builds the same architecture with random weights and downloads nothing. With ``data_root``
    the weights are read from, and downloaded to, ``<data_root>/_work/weights``."""
    if backbone not in BACKBONE_SPECS:
        raise ValueError(f"unknown backbone {backbone!r}, expected one of {BACKBONES}")
    if data_root is not None and pretrained:
        set_torch_home(data_root)
    spec = BACKBONE_SPECS[backbone]
    model = getattr(torchvision.models, spec.builder)(weights=backbone_weights(backbone) if pretrained else None)
    if backbone == "resnet50":
        model.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(model.fc.in_features, num_classes))
    elif backbone == "convnext_tiny":
        model.classifier[2] = nn.Sequential(nn.Dropout(dropout), nn.Linear(model.classifier[2].in_features, num_classes))
    else:
        model.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(model.head.in_features, num_classes))
    return model


def head_module(model: nn.Module) -> nn.Module:
    return {"resnet50": lambda m: m.fc, "convnext_tiny": lambda m: m.classifier[2], "swin_t": lambda m: m.head}[
        backbone_of(model)
    ](model)


def head_parameters(model: nn.Module) -> list[nn.Parameter]:
    return list(head_module(model).parameters())


def backbone_parameters(model: nn.Module) -> list[nn.Parameter]:
    head_ids = {id(p) for p in head_parameters(model)}
    return [p for p in model.parameters() if id(p) not in head_ids]


def gradcam_target_layer(model: nn.Module) -> nn.Module:
    """Layer whose activations Grad-CAM uses. Swin's output is NHWC: pass it through `cam_to_nchw`."""
    return model.layer4 if backbone_of(model) == "resnet50" else model.features[-1]


def cam_to_nchw(activation: torch.Tensor, model: nn.Module) -> torch.Tensor:
    return activation.permute(0, 3, 1, 2) if backbone_of(model) == "swin_t" else activation


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())

## Pretrained weights and `download-weights`

All pretrained weights live under `<data_root>/_work/weights/` (`TORCH_HOME` is pointed there before any model is
built, so runs use the stored files, and Colab can restore them from Drive). The files land in
`<TORCH_HOME>/hub/checkpoints/`. torchvision names each file `<model>-<first 8 hex of its sha256>.pth`; the check
below uses that to detect truncated files and re-download them. `download_weights` is idempotent.

In [ ]:
#| export
def weights_dir(data_root) -> Path:
    return work_dir(data_root) / "weights"


def set_torch_home(data_root, create: bool = True) -> Path:
    """Point torch's download cache at <data_root>/_work/weights (documented location of the pretrained weights)."""
    path = weights_dir(data_root)
    if create:
        path.mkdir(parents=True, exist_ok=True)
    os.environ["TORCH_HOME"] = str(path)
    return path


def weight_checkpoint_path(weights) -> Path:
    """Where torch stores the file of a torchvision weights enum member (under the current TORCH_HOME)."""
    return Path(torch.hub.get_dir()) / "checkpoints" / Path(urlparse(weights.url).path).name


def weight_file_ok(path: Path) -> bool:
    """The file exists and, when its name carries a sha256 prefix (resnet50-11ad3fa6.pth), matches it."""
    if not path.is_file() or path.stat().st_size == 0:
        return False
    match = re.search(r"-([0-9a-f]{8})\.pth$", path.name)
    if not match:
        return True
    digest = hashlib.sha256()
    with open(fs_path(path), "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest().startswith(match.group(1))


def download_weights(data_root, out=print) -> list[dict]:
    """Fetch every pretrained weight file into <data_root>/_work/weights. Safe to call repeatedly."""
    home = set_torch_home(data_root)
    out(f"download-weights: TORCH_HOME = {home}")
    rows = []
    for name, member, purpose in WEIGHT_TABLE:
        weights = torchvision.models.get_weight(member)
        path = weight_checkpoint_path(weights)
        state = "already present"
        if not weight_file_ok(path):
            if path.exists():
                path.unlink()
                state = "re-downloaded (the stored file was damaged)"
            else:
                state = "downloaded"
            weights.get_state_dict(progress=True, check_hash=True)
            if not weight_file_ok(path):
                raise ExperimentError(f"{member}: the downloaded file {path} fails its checksum")
        size = path.stat().st_size / 1e6
        out(f"  {member}: {state}\n    purpose  {purpose}\n    url      {weights.url}\n    stored   {path} ({size:.1f} MB)")
        rows.append({"name": name, "weights": member, "url": weights.url, "path": str(path), "bytes": path.stat().st_size, "state": state})
    return rows

## Metrics (T3.5)

**Macro-F1** is the primary metric (P2): F1 per class, averaged without weights, so the small Brown Spot class
counts as much as Sheath Blight. Classes that occur neither in the truth nor in the predictions of a subset are
left out of the average (never the case on the real splits; it only matters for the tiny smoke subsets).

Every evaluation reports three subsets of a split part (val or test): `white`, `field`, and `mixed` = the whole
part (about 70% white and 30% field, like the dataset). **`domain_balanced`** is the mean of the white and field
values, so that the headline number is not dominated by the easy white domain.

In [ ]:
#| export
def classification_metrics(y_true, y_pred) -> dict:
    """Accuracy, macro-F1, per-class precision / recall / F1 / support and the 5x5 confusion matrix
    (rows = true class, columns = predicted class, in CLASS_NAMES order)."""
    y_true, y_pred = np.asarray(y_true, dtype=np.int64), np.asarray(y_pred, dtype=np.int64)
    n = int(len(y_true))
    if n == 0:
        return {"n": 0, "accuracy": None, "macro_f1": None, "per_class": {}, "confusion_matrix": []}
    labels = list(range(NUM_CLASSES))
    precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
    present = sorted(set(y_true.tolist()) | set(y_pred.tolist()))
    return {
        "n": n,
        "accuracy": float((y_true == y_pred).mean()),
        "macro_f1": float(f1_score(y_true, y_pred, labels=present, average="macro", zero_division=0)),
        "per_class": {
            name: {"precision": float(precision[i]), "recall": float(recall[i]), "f1": float(f1[i]), "support": int(support[i])}
            for i, name in enumerate(CLASS_NAMES)
        },
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=labels).tolist(),
    }


def subset_mask(domains, subset: str) -> np.ndarray:
    domains = np.asarray(domains)
    return np.ones(len(domains), dtype=bool) if subset == "mixed" else domains == subset


def subset_metrics(probs: np.ndarray, labels, domains) -> dict:
    """{"white": ..., "field": ..., "mixed": ..., "domain_balanced": {"accuracy", "macro_f1"}} for one split part."""
    preds = probs.argmax(axis=1)
    result = {s: classification_metrics(labels[m], preds[m]) for s in SUBSETS for m in [subset_mask(domains, s)]}
    balanced = {}
    for key in ("accuracy", "macro_f1"):
        values = [result[s][key] for s in DOMAINS]
        balanced[key] = None if any(v is None for v in values) else float(np.mean(values))
    result["domain_balanced"] = balanced
    return result


def macro_f1(labels, preds) -> float:
    return classification_metrics(labels, preds)["macro_f1"] if len(labels) else float("nan")


@torch.no_grad()
def predict_probs(model: nn.Module, images: torch.Tensor, device, *, batch_size: int = 64, amp: bool = False) -> np.ndarray:
    """Softmax probabilities [N, 5] (float32, numpy) for a uint8 image tensor."""
    model.eval()
    out = []
    for start in range(0, len(images), batch_size):
        x = normalize_batch(images[start : start + batch_size], device)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):
            logits = model(x)
        out.append(torch.softmax(logits.float(), dim=1).cpu())
    return torch.cat(out).numpy() if out else np.zeros((0, NUM_CLASSES), dtype=np.float32)


def evaluate_model(model: nn.Module, eval_set: EvalSet, device, *, amp: bool = False) -> tuple[dict, np.ndarray]:
    """(metrics for white / field / mixed / domain_balanced, probabilities) of a model on an EvalSet."""
    probs = predict_probs(model, eval_set.images, device, amp=amp)
    return subset_metrics(probs, eval_set.labels, eval_set.domains), probs

## Training recipe (T3.4)

Fixed for all backbones (section 6.1 of the report):

| | |
|---|---|
| Stage 1, linear probe | backbone frozen (BatchNorm statistics too), 2 epochs, AdamW lr 1e-3 on the head, constant |
| Stage 2, fine-tune | all layers, 30 epochs, AdamW, lr from the split-0 sweep in {1e-4, 3e-4}, weight decay 0.05 |
| Schedule (stage 2) | 1 epoch linear warmup, then cosine decay to 0, stepped per optimiser step |
| Loss | cross-entropy, label smoothing 0.1 |
| Batch | 32; 16 with gradient accumulation 2 as the fallback (automatic on out-of-memory, or `--batch-size 16 --grad-accum 2`) |
| Precision | AMP fp16 on CUDA only; fp32 on MPS and CPU |
| Clipping | max gradient norm 1.0 |
| Checkpoint | best validation macro-F1 over `select_domains`, evaluated after every epoch (both stages); early stop after 10 stage-2 epochs without improvement |
| Seeds | `random`, `numpy`, `torch` seeded with the split seed (+ `--seed-offset`) |

Weight decay is applied to all parameters (one parameter group), the literal reading of the recipe. The
stage-1 learning rate is constant because the stage is only two epochs long. The patience counter counts
stage-2 epochs only and starts at zero when stage 2 starts.

**Learning-rate sweep (the T3.4 rule).** For each backbone, split 0 only is trained at lr 1e-4 and 3e-4
(`run --exp <ID> --sweep-lr`, outputs in `_work/runs/sweep/<ID>_lr<lr>/split_0/`); the larger *validation mixed
macro-F1* wins (`choose-lr`), and that lr is used for all five splits. Sweep runs never count as results.

In [ ]:
#| export
import contextlib

FALLBACK_BATCH = 16


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed % (2**32))
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def lr_at_step(step: int, *, base_lr: float, warmup_steps: int, total_steps: int) -> float:
    """Linear warmup over ``warmup_steps`` optimiser steps, then cosine decay to 0 at ``total_steps``."""
    if warmup_steps > 0 and step < warmup_steps:
        return base_lr * (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return base_lr * 0.5 * (1.0 + math.cos(math.pi * min(1.0, max(0.0, progress))))


class CheckpointSelector:
    """Remembers the best state by the selection metric and decides when to stop early. Only epochs that are
    passed with ``counts_for_patience=True`` (stage 2) can run the patience out."""

    def __init__(self, patience: int):
        self.patience, self.stale = patience, 0
        self.best_score, self.best_epoch, self.best_state = -math.inf, 0, None

    def update(self, score: float, epoch: int, snapshot, *, counts_for_patience: bool) -> bool:
        """Record an epoch. ``snapshot`` is a function returning the state to keep. True if it is the new best."""
        if score > self.best_score:
            self.best_score, self.best_epoch, self.best_state, self.stale = score, epoch, snapshot(), 0
            return True
        if counts_for_patience:
            self.stale += 1
        return False

    @property
    def should_stop(self) -> bool:
        return self.stale >= self.patience


def is_out_of_memory(exc: BaseException) -> bool:
    return isinstance(exc, torch.cuda.OutOfMemoryError) or (
        isinstance(exc, RuntimeError) and "out of memory" in str(exc).lower()
    )


def free_device_memory(device: str) -> None:
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()
    elif device == "mps":
        torch.mps.empty_cache()


def cpu_state_dict(model: nn.Module) -> dict:
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


@dataclass
class TrainOutcome:
    best_state: dict
    best_epoch: int
    best_score: float
    history: list[dict]
    stopped_early: bool
    seconds: float


def train_classifier(cfg: ExperimentConfig, model: nn.Module, train_ds: LeafDataset, val_set: EvalSet, device: str, *,
                     seed: int, num_workers: int = 0, mp_context=None, log=print) -> TrainOutcome:
    """The two-stage LP-FT loop. Returns the best checkpoint (by validation macro-F1 on cfg.select_domains)."""
    started = time.time()
    amp = device == "cuda"
    sampler = EpochSampler(len(train_ds), seed)
    drop_last = len(train_ds) % cfg.batch_size == 1  # a final batch of one sample breaks BatchNorm
    loader = make_loader(train_ds, cfg.batch_size, sampler=sampler, num_workers=num_workers, drop_last=drop_last,
                         pin_memory=device == "cuda", mp_context=mp_context)
    micro_steps = len(loader)
    steps_per_epoch = math.ceil(micro_steps / cfg.grad_accum)
    select_mask = np.isin(np.asarray(val_set.domains), cfg.select_domains)
    if not select_mask.any():
        raise ExperimentError(f"{cfg.exp_id}: the validation part has no images of the domains {cfg.select_domains}")
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    selector = CheckpointSelector(cfg.patience)
    history: list[dict] = []
    epoch = 0
    stopped_early = False
    head_params = head_parameters(model)

    for stage, n_epochs in (("lp", cfg.stage1_epochs), ("ft", cfg.epochs)):
        if n_epochs <= 0:
            continue
        trainable = head_params if stage == "lp" else list(model.parameters())
        for p in model.parameters():
            p.requires_grad_(False)
        for p in trainable:
            p.requires_grad_(True)
        base_lr = cfg.stage1_lr if stage == "lp" else cfg.lr
        optimizer = torch.optim.AdamW(trainable, lr=base_lr, weight_decay=cfg.weight_decay)
        total_steps = n_epochs * steps_per_epoch
        warmup_steps = cfg.warmup_epochs * steps_per_epoch if stage == "ft" else 0
        step = 0
        for _ in range(n_epochs):
            epoch += 1
            epoch_started = time.time()
            sampler.set_epoch(epoch - 1)
            train_ds.epoch = epoch - 1
            model.train() if stage == "ft" else model.eval()  # LP: frozen backbone incl. BatchNorm statistics
            if stage == "lp":
                head_module(model).train()
            loss_sum, seen, lr_now = 0.0, 0, base_lr
            optimizer.zero_grad(set_to_none=True)
            for i, (images, labels, _) in enumerate(loader):
                x = normalize_batch(images, device)
                y = labels.to(device)
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):
                    loss = F.cross_entropy(model(x), y, label_smoothing=cfg.label_smoothing)
                loss_value = loss.item()
                if not math.isfinite(loss_value):
                    raise ExperimentError(f"{cfg.exp_id}: the training loss became {loss_value} (diverged); try a lower lr")
                scaler.scale(loss / cfg.grad_accum).backward()
                loss_sum += loss_value * len(y)
                seen += len(y)
                if (i + 1) % cfg.grad_accum == 0 or i + 1 == micro_steps:
                    lr_now = base_lr if stage == "lp" else lr_at_step(step, base_lr=base_lr, warmup_steps=warmup_steps,
                                                                    total_steps=total_steps)
                    for group in optimizer.param_groups:
                        group["lr"] = lr_now
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(trainable, cfg.clip_norm)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)
                    step += 1
            probs = predict_probs(model, val_set.images, device, amp=amp)
            preds, labels_np = probs.argmax(axis=1), val_set.labels
            domains = np.asarray(val_set.domains)
            score = macro_f1(labels_np[select_mask], preds[select_mask])
            improved = selector.update(score, epoch, lambda: cpu_state_dict(model), counts_for_patience=stage == "ft")
            row = {
                "epoch": epoch, "stage": stage, "lr": lr_now, "train_loss": loss_sum / max(1, seen),
                "val_loss": float(-np.log(np.clip(probs[np.arange(len(labels_np)), labels_np], 1e-8, 1.0)).mean()),
                "val_macro_f1": score,
                "val_accuracy": float((preds[select_mask] == labels_np[select_mask]).mean()),
                "val_f1_white": macro_f1(labels_np[domains == "white"], preds[domains == "white"]),
                "val_f1_field": macro_f1(labels_np[domains == "field"], preds[domains == "field"]),
                "val_f1_mixed": macro_f1(labels_np, preds),
                "best": int(improved), "seconds": time.time() - epoch_started,
            }
            history.append(row)
            log(f"  epoch {epoch:2d}/{cfg.stage1_epochs + cfg.epochs} {stage}  lr {lr_now:.2e}  loss {row['train_loss']:.3f}  "
                f"val F1 {score:.3f}{' *' if improved else '  '} (white {row['val_f1_white']:.3f} field {row['val_f1_field']:.3f})  "
                f"{row['seconds']:.1f}s")
            if stage == "ft" and selector.should_stop:
                stopped_early = True
                log(f"  early stop: no improvement for {cfg.patience} fine-tuning epochs (best epoch {selector.best_epoch})")
                break
    return TrainOutcome(selector.best_state, selector.best_epoch, selector.best_score, history, stopped_early,
                        time.time() - started)

## Run folders and `run_split` (T3.5)

One trained model = one folder `_work/runs/<EXP>/split_<k>/`:

| File | Content |
|---|---|
| `config.json` | the resolved experiment config, which values came from `choices.json`, overrides, weights used |
| `env.json` | python / torch / torchvision / OpenCV / numpy / sklearn versions, device, GPU name |
| `history.csv` | per epoch: stage, lr, train loss, val loss, val macro-F1 on `select_domains` (and white / field / mixed), best flag, seconds |
| `predictions_val.csv`, `predictions_test.csv` | `filename, domain, true, pred, p0..p4` (class names for `true`/`pred`; `p0..p4` follow `CLASS_NAMES`) |
| `best.pt` | `{state_dict, backbone, input_mode, ...}` (gitignored; needed by `predict` and Grad-CAM) |
| `metrics.json` | split sha256, config fingerprint, wall time, val and test metrics for white / field / mixed / domain_balanced. **Written last: its existence means the run is complete.** |

Every file is written atomically (temporary file, then rename), so a Colab disconnect never leaves a
half-written file. A split is skipped when its `metrics.json` exists **and** is current (same split sha256 and
same config fingerprint); a stale one (regenerated splits, changed lr) is retrained with a visible message.
`--force` retrains regardless. The fingerprint ignores batch size, accumulation and worker count, which do not
define the experiment.

The two hook factories at the end are where phases 05 and 06 plug in: `make_extra_sources` (composites for
Method A) and `make_input_hook` (the segmentation crop for Method B). They raise `NotReadyError` until then.

In [ ]:
#| export
import io

RUN_FORMAT = 1
FINGERPRINT_IGNORED = ("batch_size", "grad_accum")
RUN_FILES = ("config.json", "env.json", "history.csv", "predictions_val.csv", "predictions_test.csv", "metrics.json")


def runs_dir(data_root) -> Path:
    return work_dir(data_root) / "runs"


def sweep_dir(data_root) -> Path:
    return runs_dir(data_root) / "sweep"


def scratch_dir(data_root) -> Path:
    return runs_dir(data_root) / "scratch"


def lr_tag(lr: float) -> str:
    """1e-4 -> '1e-4', 3e-4 -> '3e-4' (used in sweep folder names)."""
    return f"{lr:.0e}".replace("e-0", "e-").replace("e+0", "e+")


def sweep_run_name(exp_id: str, lr: float) -> str:
    return f"{exp_id}_lr{lr_tag(lr)}"


def split_dir(root: Path, split_seed: int) -> Path:
    return Path(root) / f"split_{split_seed}"


def config_fingerprint(cfg: ExperimentConfig, extra: dict | None = None) -> str:
    payload = {k: v for k, v in asdict(cfg).items() if k not in FINGERPRINT_IGNORED}
    payload.update(extra or {})
    return hashlib.sha256(json.dumps(payload, sort_keys=True, default=str).encode()).hexdigest()


def environment_record(device: str) -> dict:
    import numpy
    import sklearn

    info = {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "torch": torch.__version__,
        "torchvision": torchvision.__version__,
        "opencv": cv2.__version__,
        "numpy": numpy.__version__,
        "sklearn": sklearn.__version__,
        "device": device,
        "cpu_count": os.cpu_count(),
        "torch_home": os.environ.get("TORCH_HOME", ""),
        "gpu": None,
        "cuda": torch.version.cuda,
    }
    if device == "cuda":
        info["gpu"] = torch.cuda.get_device_name(0)
    elif device == "mps":
        info["gpu"] = "Apple MPS"
    return info


def fmt3(value) -> str:
    return "n/a" if value is None else f"{value:.3f}"


def csv_text(header: list[str], rows) -> str:
    buffer = io.StringIO()
    writer = csv.writer(buffer, lineterminator="\n")
    writer.writerow(header)
    writer.writerows(rows)
    return buffer.getvalue()


HISTORY_COLUMNS = ["epoch", "stage", "lr", "train_loss", "val_loss", "val_macro_f1", "val_accuracy",
                   "val_f1_white", "val_f1_field", "val_f1_mixed", "best", "seconds"]


def history_csv(history: list[dict]) -> str:
    return csv_text(HISTORY_COLUMNS, [[f"{row[c]:.6g}" if isinstance(row[c], float) else row[c] for c in HISTORY_COLUMNS] for row in history])


def predictions_csv(eval_set: EvalSet, probs: np.ndarray) -> str:
    preds = probs.argmax(axis=1)
    rows = [
        [name, domain, CLASS_NAMES[int(true)], CLASS_NAMES[int(pred)], *[f"{p:.6f}" for p in row]]
        for name, domain, true, pred, row in zip(eval_set.names, eval_set.domains, eval_set.labels, preds, probs)
    ]
    return csv_text(["filename", "domain", "true", "pred", *[f"p{i}" for i in range(NUM_CLASSES)]], rows)


def save_torch_atomic(obj, path: Path) -> None:
    path = Path(path)
    os.makedirs(fs_path(path.parent), exist_ok=True)
    tmp = path.with_name(path.name + f".tmp{os.getpid()}")
    with open(fs_path(tmp), "wb") as handle:
        torch.save(obj, handle)
    os.replace(fs_path(tmp), fs_path(path))


def read_metrics(run_dir: Path) -> dict | None:
    path = Path(run_dir) / "metrics.json"
    if not path.is_file():
        return None
    try:
        return json.loads(read_file_bytes(path).decode("utf-8"))
    except ValueError:
        return None  # atomic writes make this impossible in practice; treat as "not done"


def result_state(run_dir: Path, split_sha256: str, fingerprint: str) -> tuple[str, str]:
    """('fresh' | 'stale' | 'missing', reason) for the metrics.json of a run folder."""
    metrics = read_metrics(run_dir)
    if metrics is None:
        return "missing", "no metrics.json"
    if metrics.get("split_sha256") != split_sha256:
        return "stale", "the split changed since this run (split sha256 differs)"
    if metrics.get("config_fingerprint") != fingerprint:
        return "stale", "the experiment config changed since this run (config fingerprint differs)"
    return "fresh", "metrics.json is current"


def make_extra_sources(cfg: ExperimentConfig, data_root, split_seed: int, seed: int) -> tuple:
    """Extra training samples. PHASE 05 REPLACES THIS BODY: return (CompositeSource(split_seed, n_per_epoch, seed),)
    when ``cfg.use_composites``."""
    if cfg.use_composites:
        raise NotReadyError(
            f"{cfg.exp_id} trains with copy-paste composites (Method A), which need phase 05 "
            "(CompositeSource, docs/instructions/05-copy-paste-method-a.md)"
        )
    return ()


def make_input_hook(cfg: ExperimentConfig, data_root, split_seed: int):
    """``hook(image, record) -> image`` applied before the transforms. PHASE 06 REPLACES THIS BODY: return the
    segmentation-crop hook when ``cfg.seg_crop``. ``record`` is None for ad-hoc images (the predict command)."""
    if cfg.seg_crop:
        raise NotReadyError(
            f"{cfg.exp_id} crops to the segmented leaf (Method B), which needs phase 06 "
            "(docs/instructions/06-segmentation-method-b.md)"
        )
    return None


def check_cache_complete(data_root, records: list[Record]) -> None:
    missing = [r.filename for r in records if not (cache_dir(data_root) / r.filename).is_file()]
    if missing:
        raise DatasetError(f"{len(missing)} cached images are missing under {cache_dir(data_root)}, e.g. {missing[:3]}; run `prepare`")

In [ ]:
#| export
@dataclass(frozen=True)
class RunOptions:
    """How to run (as opposed to what to run): nothing here changes the experiment's result definition."""

    device: str = "cpu"
    seed_offset: int = 0
    num_workers: int | None = None  # None: default_num_workers()
    force: bool = False
    limit: int | None = None  # --fast: at most this many images per split part
    overrides: tuple = ()  # (name, value) pairs applied on top of the registry, recorded in config.json
    weights_root: Path | None = None  # where the pretrained weights live (default: the data root)
    mp_context: str | None = None  # tests only: force the worker start method


def train_with_fallback(cfg, train_ds, val_set, options: RunOptions, seed: int, log):
    """Build a fresh model and train it; on out-of-memory retry once with batch 16 and accumulation x2
    (same effective batch). Returns (model, outcome, cfg actually used)."""
    workers = default_num_workers() if options.num_workers is None else options.num_workers
    while True:
        try:
            seed_everything(seed)
            model = build_model(cfg.backbone, pretrained=cfg.pretrained, dropout=cfg.dropout,
                                data_root=options.weights_root).to(options.device)
            outcome = train_classifier(cfg, model, train_ds, val_set, options.device, seed=seed, num_workers=workers,
                                       mp_context=options.mp_context, log=log)
            return model, outcome, cfg
        except Exception as exc:
            if not (is_out_of_memory(exc) and cfg.batch_size > FALLBACK_BATCH):
                raise
            model = None
            log(f"  out of memory at batch {cfg.batch_size}: retrying with batch {FALLBACK_BATCH} and gradient accumulation "
                f"{cfg.grad_accum * cfg.batch_size // FALLBACK_BATCH}")
            cfg = replace(cfg, batch_size=FALLBACK_BATCH, grad_accum=cfg.grad_accum * cfg.batch_size // FALLBACK_BATCH)
        free_device_memory(options.device)


def run_split(resolved: "ResolvedExperiment", split_seed: int, *, data_root, out_dir: Path, options: RunOptions,
              log=print) -> dict:
    """Train and evaluate one experiment on one split and write the run folder (see the table above).

    Returns {"status": "skipped" | "done", "metrics": dict, "out_dir": Path}."""
    data_root, out_dir = Path(data_root), Path(out_dir)
    cfg = resolved.config
    split = load_split(data_root, split_seed)  # verifies the sha256
    fingerprint = config_fingerprint(cfg, {"seed_offset": options.seed_offset, "limit": options.limit})
    state, reason = result_state(out_dir, split["sha256"], fingerprint)
    if state == "fresh" and not options.force:
        log(f"[{resolved.requested} split {split_seed}] skipped: {out_dir / 'metrics.json'} exists and is current (--force to retrain)")
        return {"status": "skipped", "metrics": read_metrics(out_dir), "out_dir": out_dir}
    if state == "stale":
        log(f"[{resolved.requested} split {split_seed}] existing result is stale: {reason}; retraining")
    (out_dir / "metrics.json").unlink(missing_ok=True)  # a half-finished retrain must not look complete

    started = time.time()
    seed = split_seed + options.seed_offset
    records = load_index(data_root)
    train_recs = select_records(records, split["train"], cfg.train_domains)
    val_recs = select_records(records, split["val"])
    test_recs = select_records(records, split["test"])
    if options.limit:
        train_recs, val_recs, test_recs = (subsample_records(r, options.limit) for r in (train_recs, val_recs, test_recs))
    check_cache_complete(data_root, [*train_recs, *val_recs, *test_recs])

    extra_sources = make_extra_sources(cfg, data_root, split_seed, seed)
    input_hook = make_input_hook(cfg, data_root, split_seed)
    train_tf = ImageTransform(cfg.input_mode, train=True, size_div=cfg.size_div)
    eval_tf = ImageTransform(cfg.input_mode, train=False, size_div=cfg.size_div)
    train_ds = LeafDataset(data_root, train_recs, train_tf, seed=seed, extra_sources=extra_sources, input_hook=input_hook)
    workers = default_num_workers() if options.num_workers is None else options.num_workers
    val_set = materialize(LeafDataset(data_root, val_recs, eval_tf, input_hook=input_hook), num_workers=workers)

    log(f"[{resolved.requested} split {split_seed}] {cfg.exp_id}: {cfg.backbone} / {cfg.input_mode}  lr {cfg.lr:g}  "
        f"train {len(train_ds)} ({'+'.join(cfg.train_domains)}), val {len(val_set)}, test {len(test_recs)}  "
        f"device {options.device}, batch {cfg.batch_size}x{cfg.grad_accum}, workers {usable_workers(workers)}")
    write_json(out_dir / "config.json", {
        "format": RUN_FORMAT, "requested_exp_id": resolved.requested, "experiment": asdict(cfg),
        "split": split_seed, "seed": seed, "seed_offset": options.seed_offset, "limit": options.limit,
        "choices_used": resolved.choices_used, "notes": list(resolved.notes), "overrides": dict(options.overrides),
        "weights": BACKBONE_SPECS[cfg.backbone].weights if cfg.pretrained else None,
        "input_size": list(input_size(cfg.input_mode, cfg.size_div)), "config_fingerprint": fingerprint,
    })
    write_json(out_dir / "env.json", environment_record(options.device))

    model, outcome, used_cfg = train_with_fallback(cfg, train_ds, val_set, options, seed, log)
    model.load_state_dict(outcome.best_state)
    amp = options.device == "cuda"
    val_metrics, val_probs = evaluate_model(model, val_set, options.device, amp=amp)
    test_set = materialize(LeafDataset(data_root, test_recs, eval_tf, input_hook=input_hook), num_workers=workers)
    test_metrics, test_probs = evaluate_model(model, test_set, options.device, amp=amp)

    write_text_atomic(out_dir / "history.csv", history_csv(outcome.history))
    write_text_atomic(out_dir / "predictions_val.csv", predictions_csv(val_set, val_probs))
    write_text_atomic(out_dir / "predictions_test.csv", predictions_csv(test_set, test_probs))
    save_torch_atomic({
        "format": RUN_FORMAT, "state_dict": outcome.best_state, "backbone": cfg.backbone, "input_mode": cfg.input_mode,
        "size_div": cfg.size_div, "dropout": cfg.dropout, "class_names": list(CLASS_NAMES), "epoch": outcome.best_epoch,
        "val_select_macro_f1": outcome.best_score, "exp_id": cfg.exp_id, "split": split_seed, "config": asdict(cfg),
    }, out_dir / "best.pt")
    epoch_seconds = [row["seconds"] for row in outcome.history]
    metrics = {
        "format": RUN_FORMAT, "exp_id": cfg.exp_id, "requested_exp_id": resolved.requested, "split": split_seed,
        "seed": seed, "split_sha256": split["sha256"], "config_fingerprint": fingerprint,
        "wall_time_s": round(time.time() - started, 2), "train_time_s": round(outcome.seconds, 2),
        "seconds_per_epoch": round(float(np.mean(epoch_seconds)), 3) if epoch_seconds else None,
        "device": options.device, "backbone": cfg.backbone, "input_mode": cfg.input_mode, "lr": cfg.lr,
        "parameters": count_parameters(model), "train_domains": list(cfg.train_domains),
        "select_domains": list(cfg.select_domains), "test_subsets": list(cfg.test_subsets),
        "n_train": len(train_ds), "n_val": len(val_set), "n_test": len(test_set),
        "epochs_run": len(outcome.history), "best_epoch": outcome.best_epoch,
        "best_val_select_macro_f1": outcome.best_score, "stopped_early": outcome.stopped_early,
        "batch_size": used_cfg.batch_size, "grad_accum": used_cfg.grad_accum,
        "oom_fallback": used_cfg.batch_size != cfg.batch_size, "fast": options.limit is not None,
        "overrides": dict(options.overrides), "val": val_metrics, "test": test_metrics,
    }
    write_json(out_dir / "metrics.json", metrics, indent=1)  # last: its presence marks the run as complete
    log(f"[{resolved.requested} split {split_seed}] done in {metrics['wall_time_s']:.0f}s  best epoch {outcome.best_epoch}  "
        f"val mixed F1 {fmt3(val_metrics['mixed']['macro_f1'])}  test F1 mixed {fmt3(test_metrics['mixed']['macro_f1'])} "
        f"white {fmt3(test_metrics['white']['macro_f1'])} field {fmt3(test_metrics['field']['macro_f1'])}")
    return {"status": "done", "metrics": metrics, "out_dir": out_dir}